# 02 · Modelling: how accurately can strength be predicted?

Plan (from our "Next steps" slide):

| Step | What we do |
|---|---|
| **Baseline** | Linear regression (with scaling) |
| **Compare** | Ridge, k-nearest neighbours, Random Forest, Gradient Boosting |
| **Validate** | 80/20 train/test split + 5-fold cross-validation on the training set |
| **Explain** | Feature importance (permutation), residual analysis, predicted vs actual |

Metrics are **MAE**, **RMSE** (both in MPa) and **R²**. The test set is used **once**, at the end.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = RESULTS_DIR / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Load cleaned data and split
Run `01_data_cleaning_eda.ipynb` first, because it creates `data/concrete_clean.csv`.

In [ ]:
df = pd.read_csv(DATA_DIR / "concrete_clean.csv")
FEATURES = ["cement", "slag", "fly_ash", "water", "superplasticizer",
            "coarse_agg", "fine_agg", "age"]
TARGET = "strength"
X, y = df[FEATURES], df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows (held out until the end)")

## 2. Define the models

Scaling sits **inside** each pipeline, so during cross-validation the scaler is fitted only on that fold's training data. This prevents data leakage. Tree models don't need scaling.

In [ ]:
models = {
    "Mean baseline (dummy)": DummyRegressor(strategy="mean"),
    "Linear regression": Pipeline([("scale", StandardScaler()), ("model", LinearRegression())]),
    "Ridge regression": Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "k-NN (k=5)": Pipeline([("scale", StandardScaler()), ("model", KNeighborsRegressor(n_neighbors=5, weights="distance"))]),
    "Random forest": RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

## 3. Compare models with 5-fold cross-validation (training set only)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}

rows = []
for name, model in models.items():
    s = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({
        "model": name,
        "CV MAE": -s["test_MAE"].mean(),
        "CV RMSE": -s["test_RMSE"].mean(),
        "CV RMSE sd": s["test_RMSE"].std(),
        "CV R2": s["test_R2"].mean(),
        "CV R2 sd": s["test_R2"].std(),
    })
cv_results = pd.DataFrame(rows).set_index("model").sort_values("CV RMSE").round(3)
cv_results.to_csv(TABLE_DIR / "cv_model_comparison.csv")
cv_results

In [ ]:
order = cv_results.drop("Mean baseline (dummy)").sort_values("CV R2")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(order.index, order["CV R2"], xerr=order["CV R2 sd"], color="#4C72B0", capsize=3)
ax.set_xlabel("Mean 5-fold CV R² (± 1 sd)")
ax.set_xlim(0, 1)
for i, v in enumerate(order["CV R2"]):
    ax.text(v + 0.02, i, f"{v:.3f}", va="center")
ax.set_title("Model comparison (training set, 5-fold CV)")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_cv_model_comparison.png", bbox_inches="tight")
plt.show()

## 4. Tune the best model family

Gradient boosting usually comes out on top here. We tune a small grid **with cross-validation on the training set only**, so the test set stays untouched. If a different model wins in your run, tune that one instead.

In [ ]:
param_grid = {
    "n_estimators": [300, 600],
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 4, 5],
    "subsample": [0.8, 1.0],
}
search = GridSearchCV(GradientBoostingRegressor(random_state=RANDOM_STATE),
                      param_grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print(f"Best CV RMSE: {-search.best_score_:.3f} MPa")
best_model = search.best_estimator_

## 5. Final evaluation on the unseen test set
Every model is refitted on the full training set and scored once on the test set.

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    return pred, {
        "Test MAE": mean_absolute_error(y_te, pred),
        "Test RMSE": np.sqrt(mean_squared_error(y_te, pred)),
        "Test R2": r2_score(y_te, pred),
    }

final_models = {"Linear regression (baseline)": models["Linear regression"],
                "Random forest": models["Random forest"],
                "Gradient boosting (tuned)": best_model}

test_rows, preds = [], {}
for name, model in final_models.items():
    preds[name], m = evaluate(model, X_train, y_train, X_test, y_test)
    test_rows.append({"model": name, **m})
test_results = pd.DataFrame(test_rows).set_index("model").round(3)
test_results.to_csv(TABLE_DIR / "test_set_results.csv")
test_results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True)
lims = [0, max(y_test.max(), max(p.max() for p in preds.values())) + 5]
for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(y_test, p, s=14, alpha=0.6)
    ax.plot(lims, lims, "k--", lw=1, label="perfect prediction")
    r = test_results.loc[name]
    ax.set_title(f"{name}\nR² = {r['Test R2']:.3f} | RMSE = {r['Test RMSE']:.2f} MPa", fontsize=10)
    ax.set_xlabel("Actual strength (MPa)")
axes[0].set_ylabel("Predicted strength (MPa)")
axes[0].legend(loc="upper left")
fig.suptitle("Predicted vs actual strength on the held-out test set")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_predicted_vs_actual.png", bbox_inches="tight")
plt.show()

## 6. Residual analysis
Where does the best model go wrong? We look for patterns in the errors by predicted value and by age.

In [ ]:
best_name = "Gradient boosting (tuned)"
resid = y_test - preds[best_name]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].scatter(preds[best_name], resid, s=14, alpha=0.6)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_xlabel("Predicted strength (MPa)"); axes[0].set_ylabel("Residual (actual − predicted, MPa)")
axes[0].set_title("Residuals vs predicted")

axes[1].hist(resid, bins=30, color="#4C72B0", edgecolor="white")
axes[1].set_xlabel("Residual (MPa)"); axes[1].set_title("Residual distribution")

axes[2].scatter(X_test["age"], resid, s=14, alpha=0.6)
axes[2].axhline(0, color="k", lw=1)
axes[2].set_xscale("log"); axes[2].set_xlabel("Age (days, log)"); axes[2].set_title("Residuals vs age")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_residuals.png", bbox_inches="tight")
plt.show()

within = {t: (resid.abs() <= t).mean() * 100 for t in (2.5, 5, 10)}
for t, pct in within.items():
    print(f"{pct:5.1f}% of test predictions are within ±{t} MPa")
print(f"Largest absolute error: {resid.abs().max():.2f} MPa")

## 7. Feature importance

We use **permutation importance on the test set**: shuffle one feature and measure how much R² drops. It is model-agnostic, and unlike built-in tree importances it is not biased toward features with many unique values.

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=30,
                              random_state=RANDOM_STATE, scoring="r2", n_jobs=-1)
imp = pd.DataFrame({"importance (drop in R²)": perm.importances_mean,
                    "sd": perm.importances_std}, index=FEATURES).sort_values("importance (drop in R²)")
imp.round(4).to_csv(TABLE_DIR / "permutation_importance.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(imp.index, imp["importance (drop in R²)"], xerr=imp["sd"], color="#4C72B0", capsize=3)
ax.set_xlabel("Mean drop in test R² when feature is shuffled")
ax.set_title(f"Permutation importance: {best_name}")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_permutation_importance.png", bbox_inches="tight")
plt.show()
imp.sort_values("importance (drop in R²)", ascending=False).round(4)

In [ ]:
# Linear baseline coefficients (on standardised inputs) for comparison
lin = models["Linear regression"].fit(X_train, y_train)
coefs = pd.Series(lin.named_steps["model"].coef_, index=FEATURES).sort_values()
print("Linear regression standardised coefficients (MPa per 1 sd change):")
print(coefs.round(2))

## 8. Robustness check: does the model generalise to *new mixes*?

The same mix design often appears several times at **different ages**. A random split can therefore put a mix's 28-day sample in training and its 90-day sample in the test set, which makes the task easier. This is the representation risk from our ethics slide.

Here we group rows by mix (the 7 ingredient columns) and use **GroupKFold**, so each mix is entirely in train or entirely in test. If the scores drop, that tells us how much harder it is to predict a mix the model has never seen.

In [ ]:
groups = df.groupby(FEATURES[:7]).ngroup()
print(f"{groups.nunique()} distinct mix designs across {len(df)} rows")

gkf = GroupKFold(n_splits=5)
rows = []
for name, model in [("Linear regression", models["Linear regression"]),
                    ("Random forest", models["Random forest"]),
                    ("Gradient boosting (tuned)", best_model)]:
    s_rand = cross_validate(model, X, y, cv=cv, scoring=scoring)
    s_grp = cross_validate(model, X, y, cv=gkf, groups=groups, scoring=scoring)
    rows.append({"model": name,
                 "Random CV R2": s_rand["test_R2"].mean(), "Grouped CV R2": s_grp["test_R2"].mean(),
                 "Random CV RMSE": -s_rand["test_RMSE"].mean(), "Grouped CV RMSE": -s_grp["test_RMSE"].mean()})
robust = pd.DataFrame(rows).set_index("model").round(3)
robust.to_csv(TABLE_DIR / "grouped_vs_random_cv.csv")
robust

## 9. Optional extension: domain-informed feature

The **water/cement ratio** is the classic driver of concrete strength in civil engineering. Does adding it (plus total binder) help?

In [ ]:
def add_features(d):
    d = d.copy()
    d["binder"] = d["cement"] + d["slag"] + d["fly_ash"]
    d["water_cement_ratio"] = d["water"] / d["cement"]
    d["water_binder_ratio"] = d["water"] / d["binder"]
    return d

Xtr_fe, Xte_fe = add_features(X_train), add_features(X_test)
rows = []
for name, model in [("Linear regression", models["Linear regression"]),
                    ("Gradient boosting (tuned)", best_model)]:
    _, base = evaluate(model, X_train, y_train, X_test, y_test)
    _, fe = evaluate(model, Xtr_fe, y_train, Xte_fe, y_test)
    rows.append({"model": name, "Test R2 (8 features)": base["Test R2"], "Test R2 (+ ratios)": fe["Test R2"],
                 "Test RMSE (8 features)": base["Test RMSE"], "Test RMSE (+ ratios)": fe["Test RMSE"]})
pd.DataFrame(rows).set_index("model").round(3)

## 10. Answer to the research question

In [ ]:
b, base = test_results.loc[best_name], test_results.loc["Linear regression (baseline)"]
g = robust.loc[best_name]
summary = f"""
How accurately can concrete compressive strength be predicted from mix composition and age?

* Best model: {best_name}
    Test R²   = {b['Test R2']:.3f}  (explains ~{b['Test R2']*100:.0f}% of the variance in strength)
    Test MAE  = {b['Test MAE']:.2f} MPa
    Test RMSE = {b['Test RMSE']:.2f} MPa   (target range 2.3 to 82.6 MPa, mean ≈ {y.mean():.1f} MPa)
* Linear baseline: R² = {base['Test R2']:.3f}, RMSE = {base['Test RMSE']:.2f} MPa
    The nonlinear model cuts RMSE by {(1 - b['Test RMSE']/base['Test RMSE'])*100:.0f}%, which supports UCI's
    description of the relationship as highly nonlinear.
* {within[5]:.0f}% of test predictions are within ±5 MPa of the measured strength.
* Most important inputs (permutation): {', '.join(imp.sort_values('importance (drop in R²)', ascending=False).index[:3])}
* On completely unseen mix designs (grouped CV), R² = {g['Grouped CV R2']:.3f}, RMSE = {g['Grouped CV RMSE']:.2f} MPa.
    This is the more honest estimate for a new mix.
"""
print(summary)
(RESULTS_DIR / "summary.txt").write_text(summary, encoding="utf-8")

### Interpretation and limitations (for the write-up)

* **Accuracy:** Ensemble tree models capture the nonlinear, interacting effects of age and ingredients far better than a linear model.
* **Responsible use:** An error of several MPa still matters in structural design. The model is a **screening and decision-support tool**, not a replacement for laboratory testing, standards or engineering judgement.
* **Generalisation:** Performance drops when whole mixes are held out, and the model is only trustworthy within the range of mixes and ages in the dataset (for example, ages beyond 365 days or unusual admixtures are out of scope).
* **Data:** These are single-lab historical data from one source (Yeh, 1998). Measurement noise and lab-specific conditions limit the best achievable accuracy.